# sweet.tv — TV programme recommenderEnd-to-end walkthrough of the submitted solution: the interval join, the calibratedhold-out, the five methods required by the quiz, the LambdaRank re-ranker that wasactually shipped, and the two-user comparison.Designated Kaggle submission: `sub25_top5_s0.csv`, ref 57573516 —**public 0.45440 / private 0.44205**.Repository: https://github.com/tshermatova/sweettv-tv-program-recommenderRun every cell with `OMP_NUM_THREADS=2 VECLIB_MAXIMUM_THREADS=2` set; without it BLASoversubscribes on the small ALS solves and the machine slows by roughly a factor of ten.The heavy steps are cached under `work/`, so a second run is fast.

## 1. The data, and the one fact that shapes everythingThe view log covers weeks 11-30 and never reaches the test window, so no viewingbehaviour from weeks 31-42 exists anywhere — there is nothing to leak. The schedule forthe test weeks is supplied and is used only to know which programme airs when.**Only 10.8% of schedule rows carry a `tv_show_id`.** The other 89.2% is unlabelledairtime: it can be watched but can never be recommended.

In [ ]:
import sys, ossys.path.insert(0, "src")os.environ.setdefault("OMP_NUM_THREADS", "2")import numpy as npimport libprint(f"join events            {len(lib.UI):,}")print(f"users                  {lib.N_USERS:,}")print(f"labelled programmes    {lib.N_ITEMS:,}")print(f"test users             {len(lib.TEST_UI):,}")print()print(f"history threshold      FRAC >= {lib.FEAT_THR}")print(f"target threshold       FRAC >= {lib.TGT_THR}   (the organisers' rule)")

## 2. The joinA view is matched to a programme when they share a channel and their intervals overlap.`frac = overlap / programme duration` is capped at 1.0 by construction, because theoverlap cannot exceed the programme itself.The 0.20 threshold for building history was swept and is optimal: 0.10 and 0.35 are bothworse on both hold-outs. An absolute-seconds rule was also tried and lost — five minutesof a fifteen-minute bulletin is a third of it, five minutes of a two-hour film is theopening titles.

In [ ]:
print("frac distribution over the join")for q in (50, 75, 90, 95, 99):    print(f"   p{q}: {np.percentile(lib.FRAC, q):.3f}")print(f"   max: {lib.FRAC.max():.3f}")print()for t in (0.2, 0.5, 0.8, 1.0):    m = lib.FRAC >= t    print(f"   frac >= {t}: {m.sum():>9,} events ({100*m.mean():.2f}%)")

## 3. The target, exactly as the organisers define itFrom the competition data page:> all entries with tv_show_id = 0 were removed; for each user and each TV show user> screen time (time user watched the show divided by total duration of the show) was> calculated. Only TV shows with high screen time (>= 0.8) were taken into account; for> each user high-screen time TV shows frequency was calculated; for each user **strictly> 5 most frequent** TV shows were selected DESC - order matters!The relevant set is therefore **exactly five programmes**, not every programme watched.An earlier version of this work used `frac >= 1.0` and treated all qualifying programmesas relevant, which inflated the hold-out from 0.4316 to 0.5586 while Kaggle returned0.43499. Correcting the definition closed that gap to 0.003.

In [ ]:
GT = {u: set(v) for u, v in lib.target_top5(thr=0.80, lo=lib.B_AB, hi=lib.B_TEST).items()}sizes = [len(GT[u]) for u in lib.TEST_UI if GT.get(u)]print(f"test users with a target: {len(sizes)}/1260")print(f"median |R| = {np.median(sizes):.0f}, exactly five for {100*np.mean([s==5 for s in sizes]):.1f}%")

## 4. MAP@5Verified against the worked examples in the course slides: for relevant {A,B,C},`A,B,D,C,E` scores 0.9167 and `D,A,E,B,C` scores 0.5333. The denominator is`min(5, |relevant|)`, and a repeated prediction is counted once.

In [ ]:
R = {"A", "B", "C"}print(f"A,B,D,C,E -> {lib.apk(R, ['A','B','D','C','E'], 5):.4f}  (slides: 0.917)")print(f"D,A,E,B,C -> {lib.apk(R, ['D','A','E','B','C'], 5):.4f}  (slides: 0.533)")

## 5. CandidatesEach user's own watched programmes plus the eighty most popular. About 139 candidatesper user.Measured against the true target: recall 0.8181 and a MAP@5 ceiling of 0.8181, against amodel at 0.4395. **Ranking, not retrieval, is the binding constraint** — adding everyprogramme airing on each user's three main channels raises the ceiling to 0.8325 and themodel gains nothing (-0.0004).

In [ ]:
import importlibex = importlib.import_module("33_extra")_, CU, CI, _ = ex._base("outer")pool = {}for a, b in zip(CU, CI):    pool.setdefault(int(a), set()).add(int(lib.ITEMS_ALL[b]))tu = [u for u in lib.TEST_UI if GT.get(u)]rec = np.mean([len(GT[u] & pool.get(u, set())) / len(GT[u]) for u in tu])cap = np.mean([min(len(GT[u] & pool.get(u, set())), 5) / min(len(GT[u]), 5) for u in tu])print(f"candidates per user  {len(CU)/1260:.0f}")print(f"recall               {rec:.4f}")print(f"MAP@5 ceiling        {cap:.4f}")

## 6. The five required methodsPopularity as the baseline, then association rules, personalised PageRank, content-basedfiltering and iALS — all fitted on the history window only, in `src/methods.py`, andexposed to the ranker as four score columns.The popularity baseline is the reference every method is measured against.

In [ ]:
import featuresH = lib.history(hi=lib.B_AB)pop = lib.popularity(H)top5 = [s for s, _ in pop.most_common(5)]print("popularity top-5:", top5)print(f"popularity MAP@5 on the hold-out: {lib.evaluate({u: top5 for u in lib.TEST_UI}, GT):.5f}")

## 7. The shipped model`LGBMRanker`, objective lambdarank, 41 features, three training windows, 1600 trees atlr 0.015, rank-averaged over five seeds. The label is the organisers' own rule applied tothe training window: each user's five most frequent airings at screen time >= 0.8.That label is the single change that moved the score: +0.0079 on the hold-out and**+0.00706 on Kaggle private**. Builder: `src/36_final_feats3.py --big --seeds 5 --top5`.

In [ ]:
lab = importlib.import_module("31_lab")import features2, features3print(f"features: {len(features2.FEATS2)} base + {len(features3.FEATS3)} habit-shape = "      f"{len(features2.FEATS2)+len(features3.FEATS3)}")print()for k, v in lab.PARAMS.items():    print(f"   {k:<22} {v}")

## 8. What was tried and rejectedEleven directions were measured against the corrected target after the shippedconfiguration was fixed. Ten were neutral or worse. The table is in`report/section8.md`; the raw logs are in `work/log_*.txt`.Two results are worth reproducing here because they bound what any further work couldachieve:* expanding the pool with every programme on the user's three main channels raises the  ceiling from 0.7974 to 0.8325 and changes the score by **-0.0004**;* the top-20 of association rules adds **zero** new candidates to the pool, PageRank  zero, content-based four, iALS about zero — on this catalogue those methods and  popularity propose the same items.

## 9. The two users`src/37_two_users.py` prints each method's top five with titles and HIT/miss against theorganisers' target. The full output is in `report/section7_two_users.txt`.The result that drives Section 8: the user with **one** recorded view is predicted threetimes better than the user with 3 733, because for a sparse user the five most frequentprogrammes simply are the mass-audience bulletins.

In [ ]:
print(open("report/section7_two_users.txt").read()[:2000])